In [ ]:
!git clone https://github.com/probcomp/hfppl.git
!cd hfppl && pip install . && cd ..

In [ ]:
# This makes text wrap in the output box
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/NLP_Research_Project/"

In [1]:
import torch
import csv
from hfppl import CachedCausalLM
from smc_steer_summary import bias_model_factory, TwistModel, gen_summary
import asyncio

Load articles

In [2]:
import os
import pandas as pd

input_path = 'processed_data.csv'
df = pd.read_csv(input_path)

In [3]:
df

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center
...,...,...,...
295,Nancy Pelosi Re-Elected House Minority Leader,WASHINGTON ― House Minority Leader Nancy Pelos...,right
296,Nancy Pelosi Beats Back House Democratic Leade...,WASHINGTON — House Democrats on Wednesday reje...,center
297,Obama Will Meet With Sanders On Thursday,WASHINGTON -- With presumptive Democratic pres...,left
298,"Clinton Is 'Sane' And 'Competent,' Unlike Trum...",PHILADELPHIA ― Americans should vote for Hilla...,center


Load political bias classifier

In [ ]:
bias_model_path = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

In [9]:
async def generate_summaries(model_name, model, df, out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Title', 'Summary', 'Predicted Bias', 'Stance'])

    n_summaries = 3

    for idx, row in df.iterrows():
        title, text = row.title, row.body

        stances = ['left', 'right', 'center']

        print(f'{idx}: {title}')

        for stance in stances:
            print('------')

            for j in range(n_summaries):

              summary = await gen_summary(model_name, model, bias_model, TwistModel, text, stance)

              print('---')
              print(summary)

              pred_bias, _ = bias_model(summary)

              with open(out_path, 'a', encoding='utf-8') as f:
                  writer = csv.writer(f)
                  writer.writerow([title, summary, pred_bias, stance])

            torch.cuda.empty_cache()

            print('------------------------------------------')

# GPT 2

In [ ]:
model_name = "gpt2"
output_path = f'gpt2-smc.csv'

In [ ]:
llm = CachedCausalLM.from_pretrained(model_name)

In [ ]:
generate_summaries(model_name, llm, df, output_path)

# GPT-Neo

In [ ]:
model_name = "EleutherAI/gpt-neo-1.3B"
output_path = f'neo-smc.csv'

In [ ]:
llm = CachedCausalLM.from_pretrained(model_name)

In [ ]:
generate_summaries(model_name, llm, df, output_path)

# LLaMA 2

In [ ]:
# need to login to use llama
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"
output_path = f'llama-smc.csv'

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

llama_tokenizer = AutoTokenizer.from_pretrained(model_name, torch_dtype=torch.float32, device_map="auto")
llama_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32, device_map="auto")
llama_tokenizer.model_max_length=1024

llm = CachedCausalLM(llama_model, llama_tokenizer)

In [ ]:
generate_summaries(model_name, llm, df, output_path)